# Config

## Add project root to Python path inside the notebook

In [2]:
import sys, os

# Go one level up from the notebook folder to project root
project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root added:", project_root)

Project root added: c:\Users\atulsehgal\OneDrive\Documents\repos\talk-to-my-data-semantic


In [2]:
import inspect
from src.semantic.join_graph import JoinGraph

print(inspect.getsource(JoinGraph.find_path))

    def find_path(
        self,
        start: str,
        target: str,
        preferred_roles: Optional[Set[str]] = None,
    ) -> Optional[List[JoinEdge]]:
        """
        Role-aware join path finder.

        What this method does:
        ------------------------------------
        We want to find a join path between two tables (start → target).
        But not all join paths are equally meaningful semantically.

        Example:
            lineitem → supplier → nation → region      (shorter, but supplier region)
            lineitem → orders → customer → nation → region  (longer, but customer region)

        Business meaning prefers customer geography, not supplier geography.

        Therefore, we use TWO ranking criteria:

        1. PRIMARY RANK: number of edges whose role is in `preferred_roles`
           (e.g., {"customer_hierarchy"}). More matches = better.

        2. SECONDARY RANK: number of hops (path length).
           Among paths with the same role score, c